In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)

In [12]:
citation

{'doi': 'https://doi.org/10.1038/s41422-018-0066-y',
 'citation': 'Liu, Y., Fan, X., Wang, R. et al. Single-cell RNA-seq reveals the diversity of trophoblast subtypes and patterns of differentiation in the human placenta . Cell Res 28, 819–832 (2018)',
 'table': 'https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-018-0066-y/MediaObjects/41422_2018_66_MOESM10_ESM.xlsx'}

## Define the metadata

In [13]:
organism = "homo_sapiens"
cell_source = "placenta"
cell_state = None
table_name = "clean_degs.xlsx" # local name

## Read in file

In [14]:
df = pd.read_excel(table_name, skiprows=1).rename(columns={"cell type": "celltype"})

df

,gene,myAUC,avg_diff,power,celltype
0,INSL4,0.981,3.045510,0.962,CTB_8W_1
1,MUC15,0.977,3.238270,0.954,CTB_8W_1
2,TBX3,0.975,2.839081,0.950,CTB_8W_1
3,KRT23,0.965,2.901008,0.930,CTB_8W_1
4,SLC40A1,0.959,3.231121,0.918,CTB_8W_1
...,...,...,...,...,...
1636,TRIM10,0.708,3.009909,0.416,Blood
1637,APOM,0.708,2.275703,0.416,Blood
1638,TMSB15B,0.705,2.384987,0.410,Blood
1639,LRRC71,0.703,2.090321,0.406,Blood


## Unfiltered

In [15]:
#df = df.groupby("cell type")["gene"].apply(list)

unfiltered_df = df.rename(columns={"celltype" : "cell_type_label", "gene": "feature_name"})
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df.drop(['myAUC', 'power', 'avg_diff'], axis=1, inplace=True) # for unfiltered degs
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": None, "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result


[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'INSL4',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'INSL4',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'MUC15',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'MUC15',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'so

In [16]:
with open("evidence_unfiltered.json", "w") as f:
    json.dump(result, f, indent = 4) 

## Perform the filtering

In [17]:
col_pval = None
col_lfc = "avg_diff"
col_gene = "gene"
col_ct = "celltype"
col_rank = None
col_auc = "myAUC"
col_power = "power"

In [18]:
min_mean = 100
max_pval = 0.8
min_lfc = 3.52
max_gene_shares = 2
max_per_celltype = 20
min_auc = 0.7
min_power = 0.8

# filter by power and AUC 
dfc = df.query(f"{col_auc} >= {min_auc} & {col_power} >= {min_power} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]# .sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [19]:
markers[col_ct].value_counts()

celltype
STB_8W      20
Macro_1     20
Mes_2       20
Blood       20
Macro_2     12
Mes_1       10
CTB_8W_1     1
Name: count, dtype: int64

In [20]:
markers.groupby(col_ct)[col_gene].apply(lambda x: list(x))

celltype
Blood       [TMEM56, PKLR, MT1G4, NFE2, SPTA1, KEL, GFI1B,...
CTB_8W_1                                             [SLC1A2]
Macro_1     [RGS1, ITGAX, SERPINA1, CECR1, MSR1, LILRB4, H...
Macro_2     [VSIG4, MAF, LYVE1, F13A1, FOLR2, MRC1, CD14, ...
Mes_1       [IGFBP7, C1R, RARRES2, SPARCL1, IGFBP5, CFD3, ...
Mes_2       [WNT2, PAMR1, HSD17B2, SPON1, CXCL14, RSPO3, P...
STB_8W      [LGALS13, GH2, HOPX, PSG10P, CGB3, LINC00967, ...
Name: gene, dtype: object

In [21]:
markers.groupby(col_ct)[col_lfc].mean().sort_values()

celltype
Macro_1     3.987992
CTB_8W_1    4.001171
Blood       4.180594
Macro_2     4.231178
Mes_2       4.487878
STB_8W      4.499354
Mes_1       4.633982
Name: avg_diff, dtype: float64

## Find Ensembl ID

In [22]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [23]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=3):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [24]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json



In [26]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'SLC1A2',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'SLC1A2',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000110436'},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'STB_8W',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'LGALS13',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'STB_8W',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'LGALS13',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000105198'},
  'source': {'s

## Save evidence

In [27]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 